# 02 — Data Preparation

This notebook generates and audits the reproducible article-week dataset used by all three learning tasks. The pipeline validates raw H&M sources, aggregates complete weeks, creates leakage-safe historical features, defines next-week regression and classification targets, assigns chronological partitions, and publishes an article-level clustering table.

The supervised candidate population contains articles with at least one recorded sale in the current week. Targets represent recorded sales rather than latent demand because stock availability is unknown.

## 1. Generate or reuse the validated publication

A missing publication is generated automatically. Set `REBUILD_PIPELINE = True` to create a fresh versioned run after changing source data, feature definitions, or configuration. The pipeline never overwrites raw files, and an incomplete run cannot replace `CURRENT.json`.

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/data/process_data.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the fashion-demand-ml project.")

REBUILD_PIPELINE = False
HIGH_DEMAND_QUANTILE = 0.75
processed_dir = ROOT / "data/processed"
current_pointer = processed_dir / "CURRENT.json"
if REBUILD_PIPELINE or not current_pointer.is_file():
    subprocess.run([
        sys.executable, str(ROOT / "src/data/process_data.py"),
        "--high-demand-quantile", str(HIGH_DEMAND_QUANTILE),
    ], cwd=ROOT, check=True)
print("Current publication pointer:", current_pointer)

Current publication pointer: /Users/nataliaburguillomartin/PDRL & MLLB/fashion-demand-ml/data/processed/CURRENT.json


## 2. Load the current validated run

The publication pointer is resolved inside `data/processed`, previews are rejected, and every consumed table is checked against its recorded SHA-256 digest. Identifiers retain leading zeros and week columns are parsed as dates.

In [2]:
pointer = json.loads(current_pointer.read_text(encoding="utf-8"))
run_dir = (processed_dir / pointer["run_dir"]).resolve()
if not run_dir.is_relative_to(processed_dir.resolve()) or pointer.get("preview"):
    raise ValueError("CURRENT.json does not reference a valid complete publication.")

summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
provenance = json.loads((run_dir / "provenance.json").read_text(encoding="utf-8"))
schema = json.loads((run_dir / "schema.json").read_text(encoding="utf-8"))
checksums = json.loads((run_dir / "checksums.json").read_text(encoding="utf-8"))

def verify_checksum(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != checksums[path.name]:
        raise ValueError(f"{path.name} changed after publication.")

article_week_path = run_dir / "article_week.csv.gz"
clustering_path = run_dir / "article_clustering.csv.gz"
for published_path in [article_week_path, clustering_path]:
    verify_checksum(published_path)

model_data = pd.read_csv(
    article_week_path,
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["week", "target_week"],
)
clustering_data = pd.read_csv(
    clustering_path,
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["observation_start", "observation_end"],
)
print("Run:", run_dir.name)
display(pd.DataFrame({
    "rows": {name: details["rows"] for name, details in schema.items()},
    "columns": {name: len(details["dtypes"]) for name, details in schema.items()},
}))
display(model_data.head())

Run: 20261024T182146-1559e91a


,rows,columns
article_week,1887139,51
articles_clean,105542,25
article_clustering,77357,24


,week,article_id,sales,min_price,max_price,online_sales,mean_price,online_share,lag_1_sales,lag_2_sales,...,product_code,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,index_group_name,section_name,garment_group_name,target_high_demand
0,2018-12-17,0108775015,227,0.006339,0.008458,178,0.008321,0.784141,300,337,...,0108775,Vest top,Garment Upper body,Solid,Black,Dark,Ladieswear,Womens Everyday Basics,Jersey Basic,1
1,2018-12-17,0108775044,107,0.003034,0.008458,72,0.008233,0.672897,108,106,...,0108775,Vest top,Garment Upper body,Solid,White,Light,Ladieswear,Womens Everyday Basics,Jersey Basic,1
2,2018-12-17,0108775051,1,0.005068,0.005068,1,0.005068,1.000000,2,3,...,0108775,Vest top,Garment Upper body,Stripe,Off White,Dusty Light,Ladieswear,Womens Everyday Basics,Jersey Basic,0
3,2018-12-17,0110065001,18,0.021593,0.025407,7,0.024913,0.388889,14,31,...,0110065,Bra,Underwear,Solid,Black,Dark,Ladieswear,Womens Lingerie,"Under-, Nightwear",1
4,2018-12-17,0110065002,21,0.016932,0.025407,18,0.019374,0.857143,6,5,...,0110065,Bra,Underwear,Solid,White,Light,Ladieswear,Womens Lingerie,"Under-, Nightwear",1


## 3. Audit the processed article-week table

The checks inspect the complete table without silently dropping rows. Historical price and channel features may be missing when an article had no recorded sale in the relevant prior weeks; those missing values must later be imputed inside each training pipeline. Other missing or infinite values are treated as data-quality failures.

In [3]:
allowed_missing_columns = {
    "lag_1_mean_price", "lag_2_mean_price", "lag_3_mean_price", "lag_4_mean_price",
    "rolling_4w_mean_price", "price_change_from_4w_mean", "rolling_4w_online_share",
}
missing_by_column = model_data.isna().sum()
unexpected_missing = missing_by_column.drop(labels=list(allowed_missing_columns), errors="ignore").sum()
numeric_values = model_data.select_dtypes(include="number")
tolerance = 1e-12
issues = pd.Series({
    "Duplicate article-week rows": model_data.duplicated(["article_id", "week"]).sum(),
    "Unexpected missing values": unexpected_missing,
    "Infinite numeric values": np.isinf(numeric_values.to_numpy()).sum(),
    "Negative sales values": (model_data[["sales", "target_next_week_sales"]] < 0).any(axis=1).sum(),
    "Negative price values": (model_data[["min_price", "mean_price", "max_price"]] < 0).any(axis=1).sum(),
    "Mean price outside min-max range": (
        (model_data["mean_price"] < model_data["min_price"] - tolerance)
        | (model_data["mean_price"] > model_data["max_price"] + tolerance)
    ).sum(),
    "Invalid online share": (~model_data["online_share"].between(0, 1)).sum(),
    "Online sales above total sales": (model_data["online_sales"] > model_data["sales"]).sum(),
    "Unobserved targets": (~model_data["target_is_observed"]).sum(),
    "Unobserved history": (~model_data["history_is_observed"]).sum(),
}, dtype="int64")
display(missing_by_column[missing_by_column.gt(0)].sort_values(ascending=False).to_frame("missing"))
display(issues.to_frame("count"))
if issues.any():
    raise ValueError("Resolve the reported data-quality issues before continuing.")
display(model_data[["sales", "mean_price", "target_next_week_sales"]].describe())

,missing
lag_4_mean_price,529782
lag_3_mean_price,490922
lag_2_mean_price,450695
lag_1_mean_price,409376
rolling_4w_mean_price,162010
price_change_from_4w_mean,162010
rolling_4w_online_share,162010


,count
Duplicate article-week rows,0
Unexpected missing values,0
Infinite numeric values,0
Negative sales values,0
Negative price values,0
Mean price outside min-max range,0
Invalid online share,0
Online sales above total sales,0
Unobserved targets,0
Unobserved history,0


,sales,mean_price,target_next_week_sales
count,1.887139e+06,1.887139e+06,1.887139e+06
mean,1.467708e+01,2.821217e-02,1.383003e+01
std,3.589778e+01,2.333422e-02,3.506646e+01
min,1.000000e+00,1.016949e-04,0.000000e+00
25%,1.000000e+00,1.354237e-02,1.000000e+00
50%,4.000000e+00,2.286441e-02,3.000000e+00
75%,1.200000e+01,3.388136e-02,1.200000e+01
max,3.100000e+03,5.067797e-01,3.100000e+03


## 4. Feature engineering and temporal semantics

Every retained row has twelve complete historical weeks, a complete current week, and a complete target week. Sales lags use zero only inside verified coverage. Historical price and channel features remain missing when no price or channel observation exists. A positive `sales_trend_4w` means that the two most recent prior weeks exceed the two earlier weeks on average.

In [4]:
feature_groups = {
    "current_week": ["sales", "mean_price", "min_price", "max_price", "online_share"],
    "sales_lags": [f"lag_{lag}_sales" for lag in range(1, 13)],
    "rolling_demand": [column for column in model_data if column.startswith("rolling_") and "price" not in column and "online" not in column],
    "trend": ["sales_trend_4w"],
    "historical_price": [column for column in model_data if "mean_price" in column and column != "mean_price"] + ["price_change_from_4w_mean"],
    "historical_channel": ["rolling_4w_online_share"],
    "calendar": ["iso_year", "iso_week"],
    "product_metadata": ["product_type_name", "product_group_name", "graphical_appearance_name",
                         "colour_group_name", "perceived_colour_value_name", "index_group_name",
                         "section_name", "garment_group_name"],
}
display(pd.Series({name: len(columns) for name, columns in feature_groups.items()}, name="feature_count").to_frame())

assert (model_data["target_week"] == model_data["week"] + pd.Timedelta(weeks=1)).all()
iso_calendar = model_data["week"].dt.isocalendar()
assert (model_data["iso_year"] == iso_calendar.year).all()
assert (model_data["iso_week"] == iso_calendar.week).all()
previous_rows = model_data[["article_id", "week", "sales"]].copy()
previous_rows["week"] += pd.Timedelta(weeks=1)
previous_rows = previous_rows.rename(columns={"sales": "previous_observed_sales"})
matched_lag = model_data[["article_id", "week", "lag_1_sales"]].merge(
    previous_rows, on=["article_id", "week"], how="inner", validate="one_to_one")
assert matched_lag["lag_1_sales"].eq(matched_lag["previous_observed_sales"]).all()
print("Verified lag-1 values against", f"{len(matched_lag):,}", "consecutive article-week rows.")

,feature_count
current_week,5
sales_lags,12
rolling_demand,6
trend,1
historical_price,6
historical_channel,1
calendar,2
product_metadata,8


Verified lag-1 values against 1,459,139 consecutive article-week rows.


## 5. Audit chronological partitions

The pipeline approximates the requested 70/15/15 row allocation without splitting target weeks. Boundaries use row counts rather than target values. Optional gap weeks are reported separately and excluded from percentage denominators.

In [5]:
split_config = summary["split_configuration"]
split_order = ["train", "validation", "test"]
split_summary = model_data.groupby("split").agg(
    rows=("article_id", "size"),
    articles=("article_id", "nunique"),
    target_weeks=("target_week", "nunique"),
    first_target=("target_week", "min"),
    last_target=("target_week", "max"),
).reindex(split_order)
if split_summary.isna().any().any():
    raise ValueError("At least one chronological partition is empty.")
split_summary["actual_percentage"] = (100 * split_summary["rows"] / split_summary["rows"].sum()).round(2)
split_summary["requested_percentage"] = pd.Series(split_config["requested_percentages"])
display(split_summary)
for earlier, later in [("train", "validation"), ("validation", "test")]:
    assert split_summary.loc[earlier, "last_target"] < split_summary.loc[later, "first_target"]
assert all(model_data.groupby("target_week")["split"].nunique().eq(1))

,rows,articles,target_weeks,first_target,last_target,actual_percentage,requested_percentage
split,,,,,,,
train,1326434,77357,62,2018-12-24,2020-02-24,70.29,70.0
validation,271215,40820,15,2020-03-02,2020-06-08,14.37,15.0
test,289490,43869,14,2020-06-15,2020-09-14,15.34,15.0


## 6. Classification target

High demand is defined using the configured quantile of **positive next-week sales in training only**. The observed integer at or above that quantile becomes a fixed unit threshold and is applied unchanged to validation and test. This class-definition threshold is distinct from the probability decision threshold later chosen for a classifier.

In [6]:
classification = summary["classification_target"]
threshold = classification["threshold_units"]
expected_class = model_data["target_next_week_sales"].ge(threshold).astype("int8")
assert model_data["target_high_demand"].isin([0, 1]).all()
assert model_data["target_high_demand"].eq(expected_class).all()
train_positive = model_data.loc[
    model_data["split"].eq("train") & model_data["target_next_week_sales"].gt(0),
    "target_next_week_sales",
]
recomputed_threshold = int(max(1, train_positive.quantile(
    classification["training_quantile"], interpolation="higher")))
assert threshold == recomputed_threshold
class_summary = model_data.groupby("split").agg(
    rows=("target_high_demand", "size"),
    high_demand_rows=("target_high_demand", "sum"),
    prevalence=("target_high_demand", "mean"),
).reindex(split_order)
display(pd.Series(classification, name="value").to_frame())
display(class_summary.style.format({"prevalence": "{:.2%}"}))

,value
definition,target_next_week_sales >= threshold
threshold_units,16
training_quantile,0.75
reference_population,positive target_next_week_sales rows in train ...
prevalence_by_split,"{'test': 0.21140972054302393, 'train': 0.19924..."


,rows,high_demand_rows,prevalence
split,,,
train,1326434,264283,19.92%
validation,271215,57814,21.32%
test,289490,61201,21.14%


## 7. Leakage-safe clustering representation

The clustering table contains one row per article and uses only current and historical signals from the training period. It includes demand level, volatility, trend, activity, price, channel mix, and product metadata. Future regression and classification targets are not used. Scaling, encoding, dimensionality reduction, and cluster selection remain the responsibility of `09_unsupervised.ipynb`.

In [7]:
target_like_columns = [column for column in clustering_data if column.startswith("target_")]
assert not target_like_columns
assert not clustering_data["article_id"].duplicated().any()
assert clustering_data["observation_end"].le(
    model_data.loc[model_data["split"].eq("train"), "week"].max()).all()
clustering_summary = pd.Series({
    "articles": len(clustering_data),
    "columns": clustering_data.shape[1],
    "numeric_descriptors": clustering_data.select_dtypes(include="number").shape[1],
    "categorical_descriptors": clustering_data.select_dtypes(exclude=["number", "datetime"]).shape[1] - 1,
    "future_target_columns": len(target_like_columns),
})
display(clustering_summary.to_frame("value"))
display(clustering_data.head())

,value
articles,77357
columns,24
numeric_descriptors,12
categorical_descriptors,9
future_target_columns,0


,article_id,observation_start,observation_end,article_weeks,mean_current_week_sales,median_current_week_sales,std_current_week_sales,max_current_week_sales,mean_12w_sales,mean_12w_sales_volatility,...,mean_online_share,product_code,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,index_group_name,section_name,garment_group_name
0,0108775015,2018-12-17,2020-02-17,58,107.672414,12.5,135.691134,382,147.455460,50.304404,...,0.890152,0108775,Vest top,Garment Upper body,Solid,Black,Dark,Ladieswear,Womens Everyday Basics,Jersey Basic
1,0108775044,2018-12-17,2020-02-17,62,85.016129,20.0,95.940298,312,97.478495,35.808488,...,0.740656,0108775,Vest top,Garment Upper body,Solid,White,Light,Ladieswear,Womens Everyday Basics,Jersey Basic
2,0108775051,2018-12-17,2019-06-24,3,1.000000,1.0,0.000000,1,5.333333,5.683886,...,0.666667,0108775,Vest top,Garment Upper body,Stripe,Off White,Dusty Light,Ladieswear,Womens Everyday Basics,Jersey Basic
3,0110065001,2018-12-17,2020-02-17,62,9.661290,8.0,6.301037,28,12.313172,6.222794,...,0.282137,0110065,Bra,Underwear,Solid,Black,Dark,Ladieswear,Womens Lingerie,"Under-, Nightwear"
4,0110065002,2018-12-17,2020-02-17,54,6.648148,3.0,9.761606,46,7.308642,5.198884,...,0.674095,0110065,Bra,Underwear,Solid,White,Light,Ladieswear,Womens Lingerie,"Under-, Nightwear"


## 8. Leakage contract and published outputs

Preprocessing parameters, category encodings, scaling, feature selection, class weights, and decision thresholds must be learned inside the permitted training window. Do not use `target_next_week_sales`, `target_high_demand`, `target_week`, `split`, `target_is_observed`, or `history_is_observed` as predictors. Identifiers remain available for joins and auditing, not as continuous quantities. The final test period remains untouched until model design and hyperparameters are fixed.

The versioned publication contains:

- `article_week.csv.gz`: supervised article-week table and chronological splits.
- `article_clustering.csv.gz`: training-history article descriptors without future targets.
- `articles_clean.csv.gz`: validated product metadata.
- `summary.json`, `provenance.json`, `schema.json`, and `checksums.json`: audit and reproducibility metadata.

## Conclusion

The resulting datasets share a documented temporal contract across regression, classification, and clustering. Historical transformations and the class definition are derived without consulting future partitions, while optional missing historical price or channel observations are left for pipeline-based imputation during model fitting.